# PromoPlanet — Clasificador Automático de Imágenes v3

**Flujo de trabajo:**
1. Subís las fotos nuevas a `Entrada/` en Drive.
2. Ejecutás este notebook completo (Ctrl+F9).
3. Cada imagen se mueve a `Categoría/Subcategoría/` según su nombre.
4. Las que no se pudieron clasificar van a `Sin categoria/` para revisión manual.

**Para reclasificar archivos que están en Sin categoria:**
Moválos manualmente a la carpeta `Entrada/` y volvé a ejecutar el notebook.

In [ ]:
# ─── CELDA 1: Autenticación ──────────────────────────────────────────────────
from google.colab import auth, drive
import os

auth.authenticate_user()
drive.mount('/content/drive')
print('✅ Autenticado y Drive montado.')

In [ ]:
# ─── CELDA 2: Configuración ──────────────────────────────────────────────────

import os

RUTA_BASE       = '/content/drive/MyDrive/Catálogo PromoPlanet'
CARPETA_ENTRADA = os.path.join(RUTA_BASE, 'Entrada')
CARPETA_SIN_CAT = os.path.join(RUTA_BASE, 'Sin categoria')

CATEGORIAS = [

  # ── DRINKWARE ──────────────────────────────────────────────────────────────
  ('Drinkware', 'Termos y mates', [
    'termo', 'terere', 'tereré', 'mate', 'matero', 'matera', 'set mate',
    'kit mate', 'bombilla', 'yerbera', 'contigo',
  ]),
  ('Drinkware', 'Botellas termicas', [
    'botella termica', 'botella térmica', 'doble pared', 'vacuum',
    'tumbler termico', 'jarro termico', 'jarro térmico',
    'vaso termico', 'vaso térmico', 'sport bottle',
    'botella inox', 'botella acero',
  ]),
  ('Drinkware', 'Botellas standard', [
    'botella', 'cantimplora', 'botella aluminio', 'botella plastica',
    'botella plastico', 'botella pet', 'botella vidrio', 'hidratacion',
  ]),
  ('Drinkware', 'Tazas mugs y jarros', [
    'taza', 'mug', 'jarro', 'pocillo', 'cafetero', 'copa',
    'vaso', 'caña', 'porron', 'porrón', 'drinkware',
  ]),

  # ── ESCRITURA ──────────────────────────────────────────────────────────────
  ('Escritura', 'Escritura fina', [
    'parker', 'waterman', 'montblanc', 'cross', 'roller', 'pluma',
    'escritura fina', 'lapicera premium', 'lapicera ejecutiva',
  ]),
  ('Escritura', 'Boligrafos ecologicos', [
    'boligrafo bambu', 'boligrafo bambú', 'boligrafo madera',
    'boligrafo reciclado', 'boligrafo eco', 'lapicera eco',
    'lapicera bambu', 'lapicera madera',
  ]),
  ('Escritura', 'Boligrafos metalicos', [
    'boligrafo metal', 'boligrafo metalico', 'boligrafo acero',
    'lapicera metal', 'lapicera metalica', 'birome metal',
  ]),
  ('Escritura', 'Boligrafos plasticos', [
    'boligrafo', 'bolígrafo', 'lapicera', 'birome', 'set escritura',
  ]),
  ('Escritura', 'Lapices', [
    'lapiz', 'lápiz', 'lapices', 'lápices', 'crayón', 'crayon',
    'lapiz mecanico', 'sacapuntas', 'set dibujo', 'set escolar',
    'kit escolar', 'kit preescolar', 'set preescolar',
  ]),
  ('Escritura', 'Marcadores y resaltadores', [
    'marcador', 'resaltador', 'destacador', 'fluorescente',
    'pizarron', 'rotulador', 'plumón', 'plumon',
  ]),

  # ── INDUMENTARIA CORPORATIVA ───────────────────────────────────────────────
  ('Indumentaria corporativa', 'Gorras', [
    'gorra', 'gorro', 'piluso', 'visera', 'beanie',
  ]),
  ('Indumentaria corporativa', 'Abrigos', [
    'campera', 'buzo', 'chaleco', 'polar', 'rompeviento', 'piloto',
    'softshell', 'fleece', 'sweater', 'abrigo', 'botinero',
  ]),
  ('Indumentaria corporativa', 'Camisas', [
    'camisa', 'oxford', 'batista',
  ]),
  ('Indumentaria corporativa', 'Delantales y pecheras', [
    'delantal', 'pechera', 'mandil', 'tunica',
  ]),
  ('Indumentaria corporativa', 'Remeras y chombas', [
    'remera', 'chomba', 'camiseta', 'polo', 'musculosa',
    'dryfit', 'dry fit', 'algodon', 'algodón',
  ]),

  # ── TECNOLOGÍA ─────────────────────────────────────────────────────────────
  ('Tecnologia', 'Audio', [
    'auricular', 'parlante', 'speaker', 'bluetooth audio',
    'earphone', 'earbuds', 'headphone', 'bocina',
  ]),
  ('Tecnologia', 'Carga y conectividad', [
    'powerbank', 'power bank', 'cargador', 'cable', 'estacion multicarga',
    'estacion de carga', 'set carga', 'pendrive', 'pen drive', 'usb',
    'hub', 'adaptador', 'memoria usb', 'carga inalambrica', 'carga rapida',
    'humidificador', 'ventilador portatil',
  ]),
  ('Tecnologia', 'Accesorios para celular', [
    'soporte celular', 'anillo celular', 'pop socket', 'popsocket',
    'holder', 'holder pop', 'selfie', 'ring light',
  ]),
  ('Tecnologia', 'Accesorios de escritorio', [
    'mouse pad', 'mousepad', 'soporte notebook', 'stand notebook',
    'webcam', 'teclado', 'hub escritorio', 'lampara led',
  ]),

  # ── ESCRITORIO Y OFICINA ───────────────────────────────────────────────────
  ('Escritorio y Oficina', 'Cuadernos y agendas', [
    'cuaderno', 'agenda', 'libreta', 'anotador', 'planificador',
    'block', 'taco notas', 'memo', 'post-it', 'sticky',
  ]),
  ('Escritorio y Oficina', 'Organizadores', [
    'organizador', 'porta lapiz', 'portalapiz', 'tarjetero', 'bandeja',
    'carpeta', 'porta documento', 'archivador', 'calculadora',
    'regla', 'tijera', 'corrector', 'sello', 'set escritorio',
    'mouse pad', 'mousepad', 'soporte',
  ]),

  # ── BOLSOS Y MOCHILAS ──────────────────────────────────────────────────────
  ('Bolsos y Mochilas', 'Mochilas', [
    'mochila', 'backpack', 'portanotebook', 'porta notebook',
  ]),
  ('Bolsos y Mochilas', 'Maletines y portfolios', [
    'maletin', 'portfolio', 'portafolio', 'maletín', 'valija', 'trolley',
  ]),
  ('Bolsos y Mochilas', 'Viaje', [
    'set viaje', 'kit viaje', 'organizador viaje', 'porta documento viaje',
    'almohada viaje', 'carry on',
  ]),
  ('Bolsos y Mochilas', 'Neceseres y accesorios', [
    'necessaire', 'neceser', 'cartuchera', 'riñonera', 'billetera',
    'porta cosmetico', 'funda', 'portadoc',
  ]),
  ('Bolsos y Mochilas', 'Bolsos', [
    'bolso', 'bandolera', 'bolsa tela grande', 'bolso deportivo',
  ]),

  # ── OUTDOORS Y BIENESTAR ───────────────────────────────────────────────────
  ('Outdoors y Bienestar', 'Paraguas', [
    'paraguas', 'sombrilla',
  ]),
  ('Outdoors y Bienestar', 'Coolers y loncheras', [
    'cooler', 'lonchera', 'bolso termico', 'bolso térmico',
    'conservadora', 'lunchera',
  ]),
  ('Outdoors y Bienestar', 'Gastronomia', [
    'tabla', 'asado', 'parrilla', 'parrillero', 'parrilllero',
    'cuchillo', 'pinza', 'set parrillero', 'vino', 'descorchador',
    'sacacorcho', 'copa vino', 'decantador', 'enfriador vino',
    'sommelier', 'utensilio cocina', 'cubiertos camperos',
    'cubiertos criollos', 'cubiertos',
  ]),
  ('Outdoors y Bienestar', 'Deporte y fitness', [
    'pelota', 'banda elastica', 'yoga', 'fitness', 'gym', 'deporte',
    'running', 'runner',
  ]),
  ('Outdoors y Bienestar', 'Cuidado personal', [
    'sanitizante', 'alcohol gel', 'barbijo', 'tapaboca', 'mascarilla',
    'desinfectante', 'botiquin', 'crema', 'protector solar', 'repelente',
    'antistress', 'anti stress', 'difusor', 'aromatizante', 'vela',
    'toalla', 'toallon', 'humidificador', 'ventilador',
  ]),

  # ── ECO Y SUSTENTABLE ──────────────────────────────────────────────────────
  ('Eco y Sustentable', 'Tote bags y bolsas', [
    'tote', 'tote bag', 'bolsa algodon', 'bolsa algodón', 'ecobolsa',
    'bolsa yute', 'bolsa friselina eco', 'bolsa reciclada',
    'bolsa algodo',
  ]),
  ('Eco y Sustentable', 'Bambu y madera', [
    'bambu', 'bambú', 'madera', 'corcho', 'algarrobo', 'lenga',
  ]),
  ('Eco y Sustentable', 'Reciclados', [
    'rpet', 'reciclado', 'reciclada', 'pet reciclado', 'plastico reciclado',
    'papel reciclado', 'carton reciclado', 'eco friendly', 'ecofriendly',
    'sustentable', 'biodegradable', 'compostable',
  ]),

  # ── LLAVEROS Y ACCESORIOS ──────────────────────────────────────────────────
  ('Llaveros y Accesorios', 'Llaveros de madera', [
    'llavero madera', 'llavero bambu', 'llavero bambú', 'llavero corcho',
  ]),
  ('Llaveros y Accesorios', 'Llaveros metalicos', [
    'llavero metal', 'llavero metalico', 'llavero acero', 'llavero zinc',
  ]),
  ('Llaveros y Accesorios', 'Multiproposito', [
    'llavero abridor', 'llavero linterna', 'llavero herramienta',
    'llavero usb', 'llavero multiproposito', 'llavero navaja',
    'pin ', 'pins ', 'prendedor',
  ]),
  ('Llaveros y Accesorios', 'Llaveros plasticos', [
    'llavero', 'keychain',
  ]),

  # ── PACKAGING Y PRESENTACIÓN ───────────────────────────────────────────────
  ('Packaging y Presentacion', 'Cajas', [
    'caja regalo', 'gift box', 'caja rigida', 'caja premium',
    'caja imantada', 'caja kraft', 'caja navideña', 'cajita',
    'estuche', 'caja armable',
  ]),
  ('Packaging y Presentacion', 'Bolsas y papel', [
    'bolsa papel', 'bolsa kraft', 'papel tissue', 'papel regalo',
    'portabotella', 'porta botella', 'bolsa packaging',
    'virutilla', 'krinkle', 'packaging',
  ]),

  # ── CATEGORÍAS SIN SUBCATEGORÍAS ──────────────────────────────────────────
  ('Reconocimiento y Premios', '', [
    'trofeo', 'plaqueta', 'medalla', 'cristal', 'reconocimiento',
    'premio', 'distincion', 'distinción', 'trophy', 'award',
    'reloj ejecutivo',
  ]),
  ('Capacitacion y Eventos', '', [
    'credencial', 'lanyard', 'portanombre', 'identificador',
    'porta credencial', 'cinta evento', 'kit evento', 'congreso',
    'conferencia', 'capacitacion', 'capacitación',
  ]),
  ('Onboarding y Bienvenida', '', [
    'onboarding', 'bienvenida', 'welcome', 'kit bienvenida',
    'kit ingreso', 'nuevo ingreso', 'incorporacion', 'incorporación',
    'sticker', 'pack sticker', 'pin patrio', 'pins patrios',
  ]),
  ('Fechas Especiales', '', [
    'navidad', 'año nuevo', 'halloween', 'pascua', 'dia de la madre',
    'dia del padre', 'dia del trabajador', 'san valentin',
    'cumpleaños', 'aniversario', 'fecha especial', 'fin de año',
  ]),
]

EXTENSIONES_VALIDAS = {'.jpg', '.jpeg', '.png', '.webp', '.gif', '.bmp', '.tiff'}

print(f'✅ Configuración cargada — {len(CATEGORIAS)} reglas de clasificación.')

In [ ]:
# ─── CELDA 3: Crear estructura de carpetas ───────────────────────────────────
carpetas = set()
carpetas.add(CARPETA_ENTRADA)
carpetas.add(CARPETA_SIN_CAT)

for cat, subcat, _ in CATEGORIAS:
    if subcat:
        carpetas.add(os.path.join(RUTA_BASE, cat, subcat))
    else:
        carpetas.add(os.path.join(RUTA_BASE, cat))

for carpeta in sorted(carpetas):
    os.makedirs(carpeta, exist_ok=True)

print(f'✅ {len(carpetas)} carpetas verificadas en Drive.')

In [ ]:
# ─── CELDA 4: Clasificar y mover imágenes ───────────────────────────────────
import shutil
import unicodedata

def normalizar(texto):
    texto = texto.lower()
    texto = unicodedata.normalize('NFD', texto)
    return ''.join(c for c in texto if unicodedata.category(c) != 'Mn')

def detectar(nombre_archivo):
    n = normalizar(nombre_archivo)
    for cat, subcat, palabras in CATEGORIAS:
        for palabra in palabras:
            if normalizar(palabra) in n:
                return cat, subcat
    return None, None

def destino_dir(cat, subcat):
    if subcat:
        return os.path.join(RUTA_BASE, cat, subcat)
    return os.path.join(RUTA_BASE, cat)

def mover(origen, dir_destino):
    nombre = os.path.basename(origen)
    dst = os.path.join(dir_destino, nombre)
    if os.path.exists(dst):
        base, ext = os.path.splitext(nombre)
        i = 2
        while os.path.exists(dst):
            dst = os.path.join(dir_destino, f'{base}_{i}{ext}')
            i += 1
    shutil.move(origen, dst)

# Mover archivos de Sin categoria a Entrada para reclasificar
reclasificados = 0
if os.path.isdir(CARPETA_SIN_CAT):
    for f in os.listdir(CARPETA_SIN_CAT):
        if os.path.splitext(f)[1].lower() in EXTENSIONES_VALIDAS:
            mover(os.path.join(CARPETA_SIN_CAT, f), CARPETA_ENTRADA)
            reclasificados += 1
if reclasificados:
    print(f'↩️  {reclasificados} archivos movidos de Sin categoria a Entrada para reclasificar.')

# Procesar Entrada
archivos = [
    f for f in os.listdir(CARPETA_ENTRADA)
    if os.path.isfile(os.path.join(CARPETA_ENTRADA, f))
    and os.path.splitext(f)[1].lower() in EXTENSIONES_VALIDAS
]

if not archivos:
    print('ℹ️  No hay imágenes en la carpeta de entrada.')
else:
    print(f'📂 Procesando {len(archivos)} imagen(es)...\n')
    clasificadas, sin_cat = [], []

    for archivo in sorted(archivos):
        origen = os.path.join(CARPETA_ENTRADA, archivo)
        cat, subcat = detectar(archivo)

        if cat:
            dir1 = destino_dir(cat, subcat)
            mover(origen, dir1)
            label = f'{cat}/{subcat}' if subcat else cat
            clasificadas.append((archivo, label))
            print(f'  ✅  {archivo}')
            print(f'       → {label}')
        else:
            mover(origen, CARPETA_SIN_CAT)
            sin_cat.append(archivo)
            print(f'  ⚠️  {archivo}')
            print(f'       → Sin categoría')

    print(f'\n─────────────────────────────────────────────')
    print(f'Procesadas    : {len(archivos)}')
    print(f'Clasificadas  : {len(clasificadas)}')
    print(f'Sin categoría : {len(sin_cat)}')

    if sin_cat:
        print(f'\n⚠️  Revisá manualmente:')
        for f in sin_cat:
            print(f'   • {f}')

In [ ]:
# ─── CELDA 5: Resumen del catálogo ───────────────────────────────────────────
print('📊 Estado actual del catálogo:\n')

cats_vistas = set()
total = 0

for cat, subcat, _ in CATEGORIAS:
    carpeta = destino_dir(cat, subcat)
    if carpeta in cats_vistas or not os.path.isdir(carpeta):
        continue
    cats_vistas.add(carpeta)
    n = len([f for f in os.listdir(carpeta) if os.path.splitext(f)[1].lower() in EXTENSIONES_VALIDAS])
    total += n
    label = f'{cat} / {subcat}' if subcat else cat
    barra = '█' * min(n, 30)
    print(f'  {label:<40} {barra} {n}')

if os.path.isdir(CARPETA_SIN_CAT):
    n = len([f for f in os.listdir(CARPETA_SIN_CAT) if os.path.splitext(f)[1].lower() in EXTENSIONES_VALIDAS])
    total += n
    if n > 0:
        print(f'  {"⚠️  Sin categoría":<40} {"█" * min(n, 30)} {n}')

print(f'\n  Total: {total} imágenes')